In [1]:
import pandas as pd
import altair as alt

# Basic set up
df = pd.read_csv('../data/Bank_Marketing_Dataset.csv')
df['Subscribed'] = df['TermDepositSubscribed'].map({1: 'Yes', 0: 'No'})

In [9]:
# Viz 1: Subscription Rate by Education Level
edu_group = df.groupby('EducationLevel')['TermDepositSubscribed'].mean().reset_index()
edu_group.columns = ['EducationLevel', 'SubscriptionRate']
edu_group = edu_group[edu_group['EducationLevel'] != 'Unknown']
edu_group['SubscriptionRate'] = (edu_group['SubscriptionRate'] * 100).round(1)

selection = alt.selection_point(fields=['EducationLevel'], on='click', empty=True)

chart_edu = alt.Chart(edu_group).mark_bar().encode(
    x=alt.X('EducationLevel:N', sort='-y', title='Education Level', scale=alt.Scale(paddingInner=0.4)),
    y=alt.Y('SubscriptionRate:Q', title='Subscription Rate (%)'),
    color=alt.value('#4C78A8'),
    opacity=alt.condition(selection, alt.value(1.0), alt.value(0.35)),
    tooltip=[
        alt.Tooltip('EducationLevel:N', title='Education Level'),
        alt.Tooltip('SubscriptionRate:Q', title='Subscription Rate (%)')
    ]
).add_params(
    selection
).properties(
    title='Subscription Rate by Education Level (click a bar to filter age chart)',
    width=400,
    height=300
)

chart_edu.save('../visualizations/chart_edu.html')
chart_edu

alt.Chart(...)

In [10]:
# Viz 2: Age Distribution by Subscription Status
df['AgeBin'] = (df['Age'] // 5) * 5

age_edu = (
    df[df['EducationLevel'] != 'Unknown']
    .groupby(['EducationLevel', 'AgeBin', 'Subscribed'])
    .size()
    .reset_index(name='Count')
)

age_all = df.groupby(['AgeBin', 'Subscribed']).size().reset_index(name='Count')
age_all['EducationLevel'] = 'All'

age_edu_full = pd.concat([age_edu, age_all], ignore_index=True)

chart_age = alt.Chart(age_edu_full).transform_filter(
    alt.expr.if_(
        selection,
        alt.datum.EducationLevel == alt.datum.EducationLevel,
        alt.datum.EducationLevel == 'All'
    )
).mark_area(
    opacity=0.5,
    interpolate='monotone'
).encode(
    x=alt.X('AgeBin:Q', title='Age'),
    y=alt.Y('Count:Q', stack=None, title='Count'),
    color=alt.Color(
        'Subscribed:N',
        legend=alt.Legend(title='Subscribed'),
        scale=alt.Scale(domain=['Yes', 'No'], range=['#4C78A8', '#F58518'])
    ),
    tooltip=[
        alt.Tooltip('AgeBin:Q', title='Age'),
        alt.Tooltip('Subscribed:N', title='Subscribed'),
        alt.Tooltip('Count:Q', title='Count')
    ]
).add_params(
    selection
).properties(
    title='Age Distribution by Subscription Status (filtered by selected education)',
    width=500,
    height=300
)

chart_age.save('../visualizations/chart_age.html')
chart_age

alt.Chart(...)